# Full Dataset Evaluation on Kaggle

Self-contained notebook for evaluating 206 WAV files across CREPE, pure pYIN, and hybrid pYIN. Attach the audio dataset to the Kaggle notebook, enable GPU, then run cells in order.


In [ ]:
# CELL 1 - Check GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else "No GPU found")

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print(f"TF GPUs: {gpus}")


In [ ]:
%%bash
# CELL 2 - Install dependencies
set -e
pip install -q crepe==0.0.12
pip install -q librosa==0.10.1
pip install -q pretty_midi==0.2.10
pip install -q openpyxl==3.1.2
pip install -q resampy==0.4.2
pip install -q soundfile==0.12.1
pip install -q ffmpeg-python==0.2.0
pip install -q tqdm
# Spleeter is not used by this evaluation, but install it if Kaggle's Python/TensorFlow stack accepts it.
pip install -q spleeter==2.3.2 || echo "Spleeter install skipped; not required for this evaluation."
apt-get update -qq
apt-get install -y ffmpeg libsndfile1 -qq


In [ ]:
# CELL 3 - Full inline source code
import glob
import hashlib
import math
import os
import re
import time
from datetime import datetime, timedelta
from pathlib import Path

import crepe
import librosa
import numpy as np
import pandas as pd
import soundfile as sf
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill
from openpyxl.utils import get_column_letter

MODEL_ORDER = ["crepe", "pure_pyin", "hybrid_pyin"]
METRIC_ORDER = ["MAE_cents", "Raw_Pitch_Accuracy", "Voicing_Recall", "Voicing_False_Alarm"]
PYIN_FMIN = float(librosa.note_to_hz("C0"))
PYIN_FMAX = float(librosa.note_to_hz("B8"))
SHARP_WORDS = {
    "Csharp": "C#",
    "Dsharp": "D#",
    "Fsharp": "F#",
}

NOTE_HZ_MAP = {
    "C0": 16.35, "C1": 32.70, "C2": 65.41, "C3": 130.81,
    "C4": 261.63, "C5": 523.25, "C6": 1046.50, "C7": 2093.00,
    "C8": 4186.01,
    "C#0": 17.32, "C#1": 34.65, "C#2": 69.30, "C#3": 138.59,
    "C#4": 277.18, "C#5": 554.37, "C#6": 1108.73, "C#7": 2217.46,
    "C#8": 4434.92,
    "D0": 18.35, "D1": 36.71, "D2": 73.42, "D3": 146.83,
    "D4": 293.66, "D5": 587.33, "D6": 1174.66, "D7": 2349.32,
    "D8": 4698.63,
    "D#0": 19.45, "D#1": 38.89, "D#2": 77.78, "D#3": 155.56,
    "D#4": 311.13, "D#5": 622.25, "D#6": 1244.51, "D#7": 2489.02,
    "D#8": 4978.03,
    "E0": 20.60, "E1": 41.20, "E2": 82.41, "E3": 164.81,
    "E4": 329.63, "E5": 659.25, "E6": 1318.51, "E7": 2637.02,
    "E8": 5274.04,
    "F0": 21.83, "F1": 43.65, "F2": 87.31, "F3": 174.61,
    "F4": 349.23, "F5": 698.46, "F6": 1396.91, "F7": 2793.83,
    "F8": 5587.65,
    "F#0": 23.12, "F#1": 46.25, "F#2": 92.50, "F#3": 185.00,
    "F#4": 369.99, "F#5": 739.99, "F#6": 1479.98, "F#7": 2959.96,
    "F#8": 5919.91,
    "G0": 24.50, "G1": 49.00, "G2": 98.00, "G3": 196.00,
    "G4": 392.00, "G5": 783.99, "G6": 1567.98, "G7": 3135.96,
    "G8": 6271.93,
    "G#0": 25.96, "G#1": 51.91, "G#2": 103.83, "G#3": 207.65,
    "G#4": 415.30, "G#5": 830.61, "G#6": 1661.22, "G#7": 3322.44,
    "G#8": 6644.88,
    "A0": 27.50, "A1": 55.00, "A2": 110.00, "A3": 220.00,
    "A4": 440.00, "A5": 880.00, "A6": 1760.00, "A7": 3520.00,
    "A8": 7040.00,
    "A#0": 29.14, "A#1": 58.27, "A#2": 116.54, "A#3": 233.08,
    "A#4": 466.16, "A#5": 932.33, "A#6": 1864.66, "A#7": 3729.31,
    "A#8": 7458.62,
    "B0": 30.87, "B1": 61.74, "B2": 123.47, "B3": 246.94,
    "B4": 493.88, "B5": 987.77, "B6": 1975.53, "B7": 3951.07,
    "B8": 7902.13,
}
for _word, _symbol in SHARP_WORDS.items():
    for _octave in range(0, 9):
        _symbol_note = f"{_symbol}{_octave}"
        if _symbol_note in NOTE_HZ_MAP:
            NOTE_HZ_MAP[f"{_word}{_octave}"] = NOTE_HZ_MAP[_symbol_note]

def normalize_note(note: str) -> str:
    note = note.replace("♯", "#").replace("＃", "#").strip()
    return note[0].upper() + note[1:]

def display_sharp_names(value: str) -> str:
    text = str(value)
    for word, symbol in SHARP_WORDS.items():
        text = re.sub(word, symbol, text, flags=re.IGNORECASE)
    return text

def parse_reference(source: str):
    source = display_sharp_names(source)
    hz_match = re.search(r"([0-9]+(?:\.[0-9]+)?)\s*hz", source, flags=re.IGNORECASE)
    note_match = re.search(r"(?<![A-Za-z0-9])([A-Ga-g](?:#|b)?[0-8])(?![A-Za-z0-9])", source)
    note = normalize_note(note_match.group(1)) if note_match else None
    if hz_match:
        return float(hz_match.group(1)), note
    if note:
        if note in NOTE_HZ_MAP:
            return float(NOTE_HZ_MAP[note]), note
        try:
            return float(librosa.note_to_hz(note)), note
        except Exception:
            return None, note
    return None, None

def infer_dataset(path, dataset_root):
    path = Path(path)
    dataset_root = Path(dataset_root)
    try:
        rel_parts = path.relative_to(dataset_root).parts
    except ValueError:
        rel_parts = path.parts
    family_raw = rel_parts[0] if rel_parts else "Unknown"
    family = family_raw.replace("_", " ").title()
    if family.lower() == "Grand Piano".lower():
        family = "Grand Piano"
    elif family.lower() == "Solo Violin".lower():
        family = "Solo Violin"
    elif family.lower() == "Sine Wave".lower():
        family = "Sine Wave"
    elif family.lower() == "Solo Female Ahh".lower():
        family = "Solo Female Ahh"
    elif family.lower() == "Polyphonic".lower():
        family = "Polyphonic"
    subset = rel_parts[1] if len(rel_parts) > 2 else path.parent.name
    return family, subset

def extract_crepe_pitch(audio_path, step_size=10, model_capacity="full", confidence_threshold=0.5):
    audio_path = Path(audio_path)
    if not audio_path.exists():
        raise FileNotFoundError(f"Audio file not found: {audio_path}")
    audio, sr = librosa.load(audio_path, sr=16000, mono=True)
    audio = audio.astype(np.float32)
    time_arr, frequency, confidence, _ = crepe.predict(
        audio,
        sr,
        step_size=step_size,
        model_capacity=model_capacity,
        viterbi=True,
        verbose=0,
    )
    if confidence_threshold is not None:
        frequency = np.where(confidence >= confidence_threshold, frequency, np.nan)
    return pd.DataFrame({"time": time_arr, "frequency": frequency, "confidence": confidence})

def extract_pyin_pitch_details(audio_path, fmin=PYIN_FMIN, fmax=PYIN_FMAX):
    audio_path = Path(audio_path)
    if not audio_path.exists():
        raise FileNotFoundError(f"Audio file not found: {audio_path}")
    y, sr = librosa.load(audio_path, sr=44100, mono=True)
    f0, voiced_flag, voiced_prob = librosa.pyin(
        y,
        fmin=fmin,
        fmax=fmax,
        sr=sr,
        frame_length=8192,
        hop_length=256,
    )
    times = librosa.times_like(f0, sr=sr, hop_length=256)
    return pd.DataFrame({"time": times, "pyin_f0": f0, "voiced_prob": voiced_prob})

def fuse_pyin_crepe(pyin_df, crepe_df, fmin=PYIN_FMIN, fmax=PYIN_FMAX, confidence_threshold=0.5):
    merged = pd.merge_asof(
        pyin_df.sort_values("time"),
        crepe_df.sort_values("time"),
        on="time",
        direction="nearest",
    )
    crepe_freq = merged["frequency"].to_numpy(dtype=float)
    pyin_freq = merged["pyin_f0"].to_numpy(dtype=float)
    crepe_conf = np.clip(merged.get("confidence", pd.Series(0.0, index=merged.index)).to_numpy(dtype=float), 0.0, 1.0)
    pyin_conf = np.clip(merged.get("voiced_prob", pd.Series(0.0, index=merged.index)).to_numpy(dtype=float), 0.0, 1.0)
    crepe_valid = np.isfinite(crepe_freq) & (crepe_freq > 0.0)
    pyin_valid = np.isfinite(pyin_freq) & (pyin_freq > 0.0)

    def adaptive_threshold(conf, valid):
        valid_conf = conf[valid & np.isfinite(conf)]
        if valid_conf.size == 0:
            return confidence_threshold
        return float(np.clip(np.nanpercentile(valid_conf, 40), 0.25, 0.85))

    crepe_threshold = adaptive_threshold(crepe_conf, crepe_valid)
    pyin_threshold = adaptive_threshold(pyin_conf, pyin_valid)
    segment_candidates = np.concatenate([crepe_freq[crepe_valid], pyin_freq[pyin_valid]])
    if segment_candidates.size:
        segment_median = float(np.exp(np.nanmedian(np.log(np.maximum(segment_candidates, 1e-9)))))
    else:
        segment_median = np.nan
    if np.isfinite(segment_median) and segment_median > 0.0:
        crepe_outlier = crepe_valid & (np.abs(1200.0 * np.log2(crepe_freq / segment_median)) > 2400.0)
        pyin_outlier = pyin_valid & (np.abs(1200.0 * np.log2(pyin_freq / segment_median)) > 2400.0)
        crepe_valid = crepe_valid & ~crepe_outlier
        pyin_valid = pyin_valid & ~pyin_outlier

    hybrid_f0 = np.full(len(merged), np.nan, dtype=float)
    for i in range(len(merged)):
        c_valid = bool(crepe_valid[i])
        p_valid = bool(pyin_valid[i])
        c_reliable = c_valid and (crepe_conf[i] >= crepe_threshold)
        p_reliable = p_valid and (pyin_conf[i] >= pyin_threshold)
        if c_reliable and p_reliable:
            disagreement_cents = abs(1200.0 * np.log2(max(crepe_freq[i], pyin_freq[i]) / max(1e-9, min(crepe_freq[i], pyin_freq[i]))))
            conf_gap = abs(crepe_conf[i] - pyin_conf[i])
            if disagreement_cents <= 100.0 or conf_gap <= 0.20:
                weight_sum = crepe_conf[i] + pyin_conf[i]
                if weight_sum > 1e-9:
                    hybrid_f0[i] = ((pyin_conf[i] * pyin_freq[i]) + (crepe_conf[i] * crepe_freq[i])) / weight_sum
                else:
                    hybrid_f0[i] = np.nanmean([pyin_freq[i], crepe_freq[i]])
            else:
                hybrid_f0[i] = pyin_freq[i] if pyin_conf[i] >= crepe_conf[i] else crepe_freq[i]
        elif p_reliable:
            hybrid_f0[i] = pyin_freq[i]
        elif c_reliable:
            hybrid_f0[i] = crepe_freq[i]
        elif p_valid:
            hybrid_f0[i] = pyin_freq[i]
        elif c_valid:
            hybrid_f0[i] = crepe_freq[i]

    merged["hybrid_f0"] = np.clip(hybrid_f0, fmin, fmax)
    series = pd.Series(merged["hybrid_f0"], dtype=float)
    series = series.where(np.isfinite(series) & (series > 0.0))
    series = series.interpolate(limit=2, limit_area="inside")
    valid_mask = series.notna()
    series = series.rolling(window=7, center=True, min_periods=1).median()
    merged["hybrid_f0"] = series.where(valid_mask).clip(lower=fmin, upper=fmax)
    return merged[["time", "hybrid_f0"]]

def build_reference_grid(audio_path, hop_length=256, sr=44100):
    y, _ = librosa.load(str(audio_path), sr=sr, mono=True)
    rms = librosa.feature.rms(y=y, frame_length=2048, hop_length=hop_length)[0]
    times = librosa.frames_to_time(np.arange(len(rms)), sr=sr, hop_length=hop_length)
    if np.all(rms <= 0):
        voiced = np.zeros_like(rms, dtype=bool)
    else:
        threshold = 0.02 * float(np.max(rms))
        voiced = rms > threshold
    return pd.DataFrame({"time": times, "gt_voiced": voiced})

def align_to_reference(ref_df, pred_df, pred_col, conf_col=None):
    cols = ["time", pred_col]
    if conf_col and conf_col in pred_df.columns:
        cols.append(conf_col)
    return pd.merge_asof(ref_df.sort_values("time"), pred_df[cols].sort_values("time"), on="time", direction="nearest")

def compute_metrics(aligned_df, model, ref_hz, crepe_conf_threshold):
    pred_f0 = aligned_df["pred_f0"].to_numpy(dtype=float)
    gt_voiced = aligned_df["gt_voiced"].to_numpy(dtype=bool)
    if model == "crepe":
        pred_conf = aligned_df.get("pred_conf", pd.Series(np.zeros(len(aligned_df), dtype=float))).to_numpy(dtype=float)
        pred_voiced = np.isfinite(pred_f0) & (pred_f0 > 0) & (pred_conf >= crepe_conf_threshold)
    else:
        pred_voiced = np.isfinite(pred_f0) & (pred_f0 > 0)
    tp = int(np.sum(gt_voiced & pred_voiced))
    fn = int(np.sum(gt_voiced & ~pred_voiced))
    fp = int(np.sum(~gt_voiced & pred_voiced))
    tn = int(np.sum(~gt_voiced & ~pred_voiced))
    recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    false_alarm = fp / (fp + tn) if (fp + tn) > 0 else np.nan
    voiced_overlap = gt_voiced & pred_voiced
    if np.any(voiced_overlap):
        pred_voiced_f0 = pred_f0[voiced_overlap]
        cents_err = 1200.0 * np.log2(np.maximum(pred_voiced_f0, 1e-9) / max(ref_hz, 1e-9))
        mae_cents = float(np.mean(np.abs(cents_err)))
        median_mae_cents = float(np.median(np.abs(cents_err)))
        raw_pitch_acc = float(np.mean(np.abs(cents_err) <= 50.0))
    else:
        mae_cents = np.nan
        median_mae_cents = np.nan
        raw_pitch_acc = np.nan
    return {
        "Model": model,
        "MAE_cents": mae_cents,
        "Median_MAE_cents": median_mae_cents,
        "Raw_Pitch_Accuracy": raw_pitch_acc,
        "Voicing_Recall": recall,
        "Voicing_False_Alarm": false_alarm,
    }

def cache_key(audio_path, model, params):
    raw = str(Path(audio_path).resolve()) + "|" + model + "|" + repr(sorted(params.items()))
    return hashlib.sha1(raw.encode("utf-8")).hexdigest()

def load_or_compute_cache(cache_dir, audio_path, model, params, compute_func):
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_path = cache_dir / f"{cache_key(audio_path, model, params)}_{model}.pkl"
    if cache_path.exists():
        return pd.read_pickle(cache_path), True
    df = compute_func()
    df.to_pickle(cache_path)
    return df, False

def evaluate_audio_file(audio_path, dataset_root, crepe_conf_threshold, crepe_model_capacity, cache_dir):
    audio_path = Path(audio_path)
    ref_hz, ref_note = parse_reference(str(audio_path))
    if ref_hz is None:
        raise ValueError(f"Could not infer reference pitch from filename: {audio_path.name}")
    family, subset = infer_dataset(audio_path, dataset_root)
    crepe_raw_df, crepe_cached = load_or_compute_cache(
        cache_dir,
        audio_path,
        "crepe",
        {"capacity": crepe_model_capacity, "threshold": None, "step_size": 10},
        lambda: extract_crepe_pitch(str(audio_path), confidence_threshold=None, model_capacity=crepe_model_capacity),
    )
    pyin_details_df, pyin_cached = load_or_compute_cache(
        cache_dir,
        audio_path,
        "pyin",
        {"fmin": PYIN_FMIN, "fmax": PYIN_FMAX, "frame_length": 8192, "hop_length": 256},
        lambda: extract_pyin_pitch_details(str(audio_path), fmin=PYIN_FMIN, fmax=PYIN_FMAX),
    )
    crepe_df = crepe_raw_df.copy()
    crepe_df["frequency"] = np.where(crepe_df["confidence"].to_numpy(dtype=float) >= crepe_conf_threshold, crepe_df["frequency"].to_numpy(dtype=float), np.nan)
    pyin_df = pyin_details_df.rename(columns={"pyin_f0": "hybrid_f0"})[["time", "hybrid_f0"]]
    hybrid_df = fuse_pyin_crepe(pyin_details_df, crepe_raw_df, fmin=PYIN_FMIN, fmax=PYIN_FMAX)
    ref_df = build_reference_grid(audio_path)
    model_frames = {
        "crepe": (crepe_df.rename(columns={"frequency": "pred_f0", "confidence": "pred_conf"}), "pred_conf"),
        "pure_pyin": (pyin_df.rename(columns={"hybrid_f0": "pred_f0"}), None),
        "hybrid_pyin": (hybrid_df.rename(columns={"hybrid_f0": "pred_f0"}), None),
    }
    rows = []
    for model in MODEL_ORDER:
        pred_df, conf_col = model_frames[model]
        aligned = align_to_reference(ref_df, pred_df, "pred_f0", conf_col)
        metrics = compute_metrics(aligned, model, ref_hz, crepe_conf_threshold)
        try:
            rel_path = str(audio_path.relative_to(Path('/kaggle/input')))
        except ValueError:
            rel_path = str(audio_path)
        metrics.update({
            "Dataset": family,
            "Subset": subset,
            "File": audio_path.name,
            "Reference_Note": ref_note or "",
            "Reference_Hz": ref_hz,
            "Path": rel_path,
            "CREPE_Cached": crepe_cached,
            "PYIN_Cached": pyin_cached,
        })
        rows.append(metrics)
    return rows

def summarize(results_df, metric, agg):
    table = results_df.pivot_table(index=["Dataset", "Subset"], columns="Model", values=metric, aggfunc=agg).reset_index()
    cols = ["Dataset", "Subset", *[m for m in MODEL_ORDER if m in table.columns]]
    return table[cols]

def best_per_note(results_df):
    rows = []
    for keys, group in results_df.groupby(["Dataset", "Subset", "File", "Reference_Note", "Reference_Hz"], dropna=False):
        valid = group.dropna(subset=["MAE_cents"])
        if valid.empty:
            best_model = ""
            best_mae = np.nan
        else:
            best = valid.loc[valid["MAE_cents"].idxmin()]
            best_model = best["Model"]
            best_mae = best["MAE_cents"]
        rows.append({
            "Dataset": keys[0],
            "Subset": keys[1],
            "File": keys[2],
            "Reference_Note": keys[3],
            "Reference_Hz": keys[4],
            "Best_Model": best_model,
            "Best_MAE_cents": best_mae,
        })
    return pd.DataFrame(rows).sort_values(["Dataset", "Subset", "Reference_Hz", "File"])

def safe_sheet_name(name, used):
    name = display_sharp_names(name)
    cleaned = re.sub(r"[\[\]:*?/\\]", " ", name).strip()[:31] or "Sheet"
    candidate = cleaned
    i = 2
    while candidate in used:
        suffix = f" {i}"
        candidate = f"{cleaned[:31 - len(suffix)]}{suffix}"
        i += 1
    used.add(candidate)
    return candidate

def find_old_results_workbook():
    candidates = glob.glob('/kaggle/input/**/Alpha_Testing_3_10_26__4_.xlsx', recursive=True)
    return Path(candidates[0]) if candidates else None

def add_old_vs_new_sheet(writer, old_path):
    if old_path is None or not Path(old_path).exists():
        pd.DataFrame([{"Status": "Old results workbook not attached", "Expected_File": "Alpha_Testing_3_10_26__4_.xlsx"}]).to_excel(writer, sheet_name="Old vs New Hybrid", index=False)
        return
    old_sheets = pd.read_excel(old_path, sheet_name=None)
    old_rows = []
    for sheet_name, sheet_df in old_sheets.items():
        sheet_df = sheet_df.copy()
        sheet_df.insert(0, "Old_Workbook_Sheet", sheet_name)
        old_rows.append(sheet_df)
    old_df = pd.concat(old_rows, ignore_index=True) if old_rows else pd.DataFrame()
    old_df.to_excel(writer, sheet_name="Old vs New Hybrid", index=False)

def style_workbook(path):
    wb = load_workbook(path)
    header_fill = PatternFill("solid", fgColor="D9EAF7")
    best_fill = PatternFill("solid", fgColor="C6EFCE")
    worst_fill = PatternFill("solid", fgColor="FFC7CE")
    for ws in wb.worksheets:
        for cell in ws[1]:
            cell.font = Font(bold=True)
            cell.fill = header_fill
        headers = [cell.value for cell in ws[1]]
        model_cols = [headers.index(model) + 1 for model in MODEL_ORDER if model in headers]
        lower_is_better = "Accuracy" not in ws.title
        if model_cols:
            for row in range(2, ws.max_row + 1):
                values = []
                for col in model_cols:
                    value = ws.cell(row=row, column=col).value
                    if isinstance(value, (int, float)) and np.isfinite(value):
                        values.append((col, value))
                if len(values) >= 2:
                    target_best = min(v for _, v in values) if lower_is_better else max(v for _, v in values)
                    target_worst = max(v for _, v in values) if lower_is_better else min(v for _, v in values)
                    for col, value in values:
                        if value == target_best:
                            ws.cell(row=row, column=col).fill = best_fill
                        if value == target_worst:
                            ws.cell(row=row, column=col).fill = worst_fill
        for col_idx, column_cells in enumerate(ws.columns, start=1):
            max_len = 0
            for cell in column_cells:
                value = "" if cell.value is None else str(cell.value)
                max_len = max(max_len, min(len(value), 80))
            ws.column_dimensions[get_column_letter(col_idx)].width = max(10, max_len + 2)
    wb.save(path)

def write_results_workbook(results_df, output_path, old_path=None):
    excel_df = results_df.copy()
    for col in ["Dataset", "Subset", "File", "Reference_Note"]:
        if col in excel_df.columns:
            excel_df[col] = excel_df[col].map(display_sharp_names)
    used = set()
    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        sheets = {
            "Summary - Median MAE": summarize(excel_df, "MAE_cents", "median"),
            "Summary - Mean MAE": summarize(excel_df, "MAE_cents", "mean"),
            "Summary - Std Dev MAE": summarize(excel_df, "MAE_cents", "std"),
            "Summary - Raw Pitch Accuracy": summarize(excel_df, "Raw_Pitch_Accuracy", "mean"),
        }
        for sheet_name, df in sheets.items():
            used.add(sheet_name)
            df.to_excel(writer, sheet_name=sheet_name, index=False)
        for dataset in ["Grand Piano", "Solo Violin", "Sine Wave", "Solo Female Ahh", "Polyphonic"]:
            group = excel_df[excel_df["Dataset"] == dataset].copy()
            if group.empty:
                continue
            if dataset == "Polyphonic":
                group.insert(0, "Evaluation_Label", "Robustness test")
            sheet_name = safe_sheet_name(f"{dataset} - Full Results", used)
            group.sort_values(["Subset", "Reference_Hz", "File", "Model"]).to_excel(writer, sheet_name=sheet_name, index=False)
        used.add("All Datasets - Combined")
        excel_df.sort_values(["Dataset", "Subset", "Reference_Hz", "File", "Model"]).to_excel(writer, sheet_name="All Datasets - Combined", index=False)
        used.add("Model Comparison - Best Per Note")
        best_per_note(excel_df).to_excel(writer, sheet_name="Model Comparison - Best PerNote", index=False)
        add_old_vs_new_sheet(writer, old_path)
    style_workbook(output_path)
    return output_path


In [ ]:
# CELL 4 - Configure paths for Kaggle
import os
import glob
from pathlib import Path

# If your Kaggle dataset slug differs, this auto-detects the folder containing the most WAV files.
preferred = Path("/kaggle/input/audio-to-midi-dataset")
if preferred.exists():
    DATASET_BASE = str(preferred)
else:
    candidates = []
    for folder in Path("/kaggle/input").glob("*"):
        if folder.is_dir():
            count = len(list(folder.rglob("*.wav")))
            if count:
                candidates.append((count, folder))
    if not candidates:
        raise FileNotFoundError("No WAV files found under /kaggle/input. Attach the dataset first.")
    DATASET_BASE = str(sorted(candidates, reverse=True)[0][1])

all_wav_files = sorted(glob.glob(DATASET_BASE + "/**/*.wav", recursive=True))
print(f"Dataset base: {DATASET_BASE}")
print(f"Found {len(all_wav_files)} WAV files")
if len(all_wav_files) != 206:
    print("Warning: expected 206 WAV files. Check attached dataset if this count is surprising.")

OUTPUT_DIR = "/kaggle/working/"
CHECKPOINT_PATH = OUTPUT_DIR + "Full_Evaluation_Checkpoint.csv"
ERROR_LOG_PATH = OUTPUT_DIR + "Full_Evaluation_Errors.csv"
CACHE_DIR = OUTPUT_DIR + "pitch_cache"
EXCEL_PATH = OUTPUT_DIR + "Full_Evaluation_Results.xlsx"

# Mapping is already defined in Cell 3 as NOTE_HZ_MAP.
print(f"Checkpoint path: {CHECKPOINT_PATH}")


In [ ]:
# CELL 5 - GPU optimization for CREPE
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except Exception as exc:
            print(f"Could not set memory growth for {gpu}: {exc}")
    print(f"GPU enabled: {gpus[0].name}")
    CREPE_CAPACITY = "full"
    BATCH_SIZE = 8
else:
    print("CPU only mode")
    CREPE_CAPACITY = "small"
    BATCH_SIZE = 1

CREPE_CONF_THRESHOLD = 0.5
print(f"CREPE capacity: {CREPE_CAPACITY}")
print(f"Batch size hint: {BATCH_SIZE}")


In [ ]:
# CELL 6 - Run full evaluation with progress, checkpointing, ETA, and keepalive
import threading
import time
from datetime import datetime, timedelta
from pathlib import Path

from tqdm.auto import tqdm

RUN_STARTED_AT = time.time()

def kaggle_keepalive():
    while True:
        time.sleep(300)
        print(f"[Keepalive] Still running... {time.strftime('%H:%M:%S')}", flush=True)

keepalive_thread = threading.Thread(target=kaggle_keepalive, daemon=True)
keepalive_thread.start()

checkpoint_path = Path(CHECKPOINT_PATH)
error_log_path = Path(ERROR_LOG_PATH)
rows = []
errors = []
completed_paths = set()

if checkpoint_path.exists():
    checkpoint_df = pd.read_csv(checkpoint_path)
    rows = checkpoint_df.to_dict('records')
    completed_paths = set(checkpoint_df['Path'].dropna().astype(str).unique()) if 'Path' in checkpoint_df.columns else set()
    print(f"Resuming from checkpoint: {len(completed_paths)} completed files, {len(rows)} model rows")

if error_log_path.exists():
    errors = pd.read_csv(error_log_path).to_dict('records')

pending_files = []
for wav_path in all_wav_files:
    try:
        rel_path = str(Path(wav_path).relative_to(Path('/kaggle/input')))
    except ValueError:
        rel_path = str(wav_path)
    if rel_path not in completed_paths:
        pending_files.append(wav_path)

print(f"Total files: {len(all_wav_files)}")
print(f"Already completed: {len(completed_paths)}")
print(f"Pending: {len(pending_files)}")

file_durations = []
pbar = tqdm(pending_files, desc="Evaluating WAV files", unit="file")
for idx, wav_path in enumerate(pbar, start=1):
    file_start = time.time()
    wav_path_obj = Path(wav_path)
    family, subset = infer_dataset(wav_path_obj, Path(DATASET_BASE))
    pbar.set_postfix_str(f"{family} | {wav_path_obj.name[:38]}")
    try:
        file_rows = evaluate_audio_file(
            wav_path_obj,
            Path(DATASET_BASE),
            CREPE_CONF_THRESHOLD,
            CREPE_CAPACITY,
            CACHE_DIR,
        )
        elapsed = time.time() - file_start
        for row in file_rows:
            row["File_Seconds"] = elapsed
            row["Completed_At"] = datetime.now().isoformat(timespec="seconds")
        rows.extend(file_rows)
        pd.DataFrame(rows).to_csv(checkpoint_path, index=False)
        file_durations.append(elapsed)
        avg = sum(file_durations) / len(file_durations)
        remaining = len(pending_files) - idx
        eta = datetime.now() + timedelta(seconds=avg * remaining)
        print(
            f"[{idx}/{len(pending_files)}] OK {wav_path_obj.name} | "
            f"{elapsed:.1f}s | avg {avg:.1f}s/file | ETA {eta.strftime('%Y-%m-%d %H:%M:%S')}",
            flush=True,
        )
    except Exception as exc:
        elapsed = time.time() - file_start
        err = {
            "Path": str(wav_path_obj),
            "Dataset": family,
            "Subset": subset,
            "File": wav_path_obj.name,
            "Error": str(exc),
            "File_Seconds": elapsed,
            "Failed_At": datetime.now().isoformat(timespec="seconds"),
        }
        errors.append(err)
        pd.DataFrame(errors).to_csv(error_log_path, index=False)
        print(f"FAILED {wav_path_obj.name}: {exc}", flush=True)

total_runtime_seconds = time.time() - RUN_STARTED_AT
print(f"Finished Cell 6 in {total_runtime_seconds/60:.2f} minutes")
print(f"Checkpoint saved to: {checkpoint_path}")
if errors:
    print(f"Errors saved to: {error_log_path} ({len(errors)} failed files)")


In [ ]:
# CELL 7 - Generate Excel results
from pathlib import Path

checkpoint_path = Path(CHECKPOINT_PATH)
if not checkpoint_path.exists():
    raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}. Run Cell 6 first.")

results_df = pd.read_csv(checkpoint_path)
print(f"Loaded {len(results_df)} model rows from checkpoint")
print(f"Unique files evaluated: {results_df['Path'].nunique() if 'Path' in results_df.columns else 'unknown'}")

old_workbook = find_old_results_workbook()
if old_workbook:
    print(f"Found old comparison workbook: {old_workbook}")
else:
    print("Old comparison workbook not attached; Old vs New sheet will note this.")

write_results_workbook(results_df, EXCEL_PATH, old_workbook)
print(f"Saved Excel results to: {EXCEL_PATH}")

try:
    from IPython.display import FileLink, display
    display(FileLink(EXCEL_PATH))
except Exception as exc:
    print(f"File link unavailable: {exc}")


In [ ]:
# CELL 8 - Print final summary and download Excel
import pandas as pd
from pathlib import Path

results_df = pd.read_csv(CHECKPOINT_PATH)
summary = results_df.groupby(["Dataset", "Model"], as_index=False)["MAE_cents"].median()
winners = []
for dataset, group in summary.groupby("Dataset"):
    valid = group.dropna(subset=["MAE_cents"])
    if valid.empty:
        winners.append({"Dataset": dataset, "Winning_Model": "N/A", "Median_MAE_cents": None})
    else:
        best = valid.loc[valid["MAE_cents"].idxmin()]
        winners.append({"Dataset": dataset, "Winning_Model": best["Model"], "Median_MAE_cents": best["MAE_cents"]})
winners_df = pd.DataFrame(winners).sort_values("Dataset")
print("Model winner per dataset, using lowest median MAE:")
display(winners_df)

old_workbook = find_old_results_workbook()
if old_workbook:
    print(f"Old hybrid workbook was attached and copied into Excel comparison sheet: {old_workbook}")
else:
    print("Old hybrid comparison workbook was not attached. Attach Alpha_Testing_3_10_26__4_.xlsx as a second Kaggle dataset if needed.")

processed = results_df["Path"].nunique() if "Path" in results_df.columns else 0
failed = 0
if Path(ERROR_LOG_PATH).exists():
    failed = pd.read_csv(ERROR_LOG_PATH)["Path"].nunique()
total_runtime = None
if "File_Seconds" in results_df.columns:
    per_file = results_df.groupby("Path")["File_Seconds"].first()
    total_runtime = float(per_file.sum())
print(f"Files processed successfully: {processed}")
print(f"Files failed: {failed}")
if total_runtime is not None:
    print(f"Approx total active evaluation time: {total_runtime/60:.2f} minutes")
print(f"Excel output: {EXCEL_PATH}")

try:
    from IPython.display import FileLink, display
    display(FileLink(EXCEL_PATH))
except Exception:
    pass

try:
    from IPython.display import Javascript, display
    display(Javascript(f"window.open('/kaggle/working/{Path(EXCEL_PATH).name}', '_blank')"))
except Exception:
    pass
